In [1]:
# Celda 1 - Librerias  [V0.6 - GP Integrado]
import cv2
import numpy as np
import time
import math
import datetime
import threading
import collections
import customtkinter as ctk
from PIL import Image

# Librerias para Optimizador DEAP
import random
from deap import base, creator, tools, algorithms

ctk.set_appearance_mode('dark')
print('Celda 1 V0.6: Librerias base y DEAP listas.')

Celda 1 V0.6: Librerias base y DEAP listas.


In [2]:
# Celda 2 - Motor de vision con Mallas YOLO y HoughCircles  [V0.6]
from pygrabber.dshow_graph import FilterGraph

def detectar_camaras_sistema():
    try:
        graph = FilterGraph()
        return [(i, n) for i, n in enumerate(graph.get_input_devices())]
    except Exception as e:
        print(f'Error al buscar camaras: {e}')
        return []

RANGO_ROJO_1 = (np.array([  0, 100,  60]), np.array([ 12, 255, 255]))
RANGO_ROJO_2 = (np.array([168, 100,  60]), np.array([180, 255, 255]))
RANGO_NEGRO  = (np.array([  0,   0,   0]), np.array([180, 255,  60]))
RANGO_BLANCO = (np.array([  0,   0, 170]), np.array([180,  45, 255]))

HOUGH_DP       = 1.2
HOUGH_MINDIST  = 60
HOUGH_PARAM1   = 120
HOUGH_PARAM2   = 55
HOUGH_MINR     = 35
HOUGH_MAXR     = 320
FRACCION_COLOR_MIN = 0.20
UMBRAL_MISMO_OBJETO = 0.82

def clasificar_color_circulo(hsv_frame, cx, cy, radio):
    h, w = hsv_frame.shape[:2]
    mascara = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mascara, (cx, cy), radio, 255, -1)
    total_px = cv2.countNonZero(mascara)
    if total_px == 0: return None
    m_rojo = cv2.add(cv2.inRange(hsv_frame, *RANGO_ROJO_1), cv2.inRange(hsv_frame, *RANGO_ROJO_2))
    m_negro  = cv2.inRange(hsv_frame, *RANGO_NEGRO)
    m_blanco = cv2.inRange(hsv_frame, *RANGO_BLANCO)
    fracs = {
        'Rojo':   cv2.countNonZero(cv2.bitwise_and(m_rojo,  mascara)) / total_px,
        'Negro':  cv2.countNonZero(cv2.bitwise_and(m_negro, mascara)) / total_px,
        'Blanco': cv2.countNonZero(cv2.bitwise_and(m_blanco,mascara)) / total_px,
    }
    mejor = max(fracs, key=fracs.get)
    if fracs[mejor] >= FRACCION_COLOR_MIN: return mejor
    return None

def calcular_huella_hsv(hsv_frame, cx, cy, radio):
    h, w = hsv_frame.shape[:2]
    mascara = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mascara, (cx, cy), radio, 255, -1)
    if cv2.countNonZero(mascara) < 50: return None
    hist = cv2.calcHist([hsv_frame], [0, 1], mascara, [30, 32], [0, 180, 0, 256])
    cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
    return hist

def es_mismo_objeto(huella_nueva, huellas_guardadas):
    if huella_nueva is None or not huellas_guardadas: return False
    for ref in huellas_guardadas:
        if cv2.compareHist(huella_nueva, ref, cv2.HISTCMP_CORREL) >= UMBRAL_MISMO_OBJETO: return True
    return False

def get_color_bgr(nombre):
    n = nombre.lower()
    if 'rojo'   in n or 'red'   in n: return (0,   0, 220)
    if 'blanco' in n or 'white' in n: return (200, 200, 200)
    if 'negro'  in n or 'black' in n: return (80,  80,  80)
    return (0, 220, 220)

def estimar_z(radio_px, frame_shape):
    frac = (math.pi * radio_px * radio_px) / (frame_shape[0] * frame_shape[1])
    return round(max(0.1, min(5.0, 1.0 / (frac * 10 + 0.01))), 2)

class TrackerCirculo:
    def __init__(self, alpha=0.35, frames_conf=2, frames_perdida=5):
        self.alpha          = alpha
        self.frames_conf    = frames_conf
        self.frames_perdida = frames_perdida
        self.reiniciar()

    def reiniciar(self):
        self.suave       = None
        self.conteo_det  = 0
        self.conteo_perd = 0
        self.visible     = False

    def actualizar(self, deteccion):
        if deteccion is None:
            self.conteo_det  = 0
            self.conteo_perd = min(self.conteo_perd + 1, self.frames_perdida + 1)
            if self.conteo_perd >= self.frames_perdida:
                self.visible = False
                self.suave   = None
            return tuple(int(round(v)) for v in self.suave) if self.visible else None
        self.conteo_perd = 0
        self.conteo_det  = min(self.conteo_det + 1, self.frames_conf + 10)
        if self.conteo_det < self.frames_conf: return None
        if self.suave is None:
            self.suave   = tuple(float(v) for v in deteccion)
            self.visible = True
            return tuple(int(round(v)) for v in self.suave)
        self.visible = True
        a = self.alpha
        self.suave = tuple(a * d + (1 - a) * s for d, s in zip(deteccion, self.suave))
        return tuple(int(round(v)) for v in self.suave)

def procesar_frame_vision(frame, temporizadores, trackers, callback_objeto=None, yolo_model=None, imgsz=640, conf=0.45):
    t_act       = time.time()
    hay_objetos = False
    fh, fw      = frame.shape[:2]

    if yolo_model is not None:
        results = yolo_model.predict(frame, imgsz=imgsz, conf=conf, verbose=False)
        yolo_dets = {}
        if results and results[0].masks is not None:
            boxes = results[0].boxes
            masks = results[0].masks.xy
            for i in range(len(boxes)):
                cls_id = int(boxes.cls[i])
                nombre = yolo_model.names[cls_id]
                x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy()
                cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
                r  = int(max(x2 - x1, y2 - y1) / 2)
                pts = np.array(masks[i], dtype=np.int32)
                yolo_dets.setdefault(nombre, []).append((cx, cy, r, float(boxes.conf[i]), pts))

        for nombre in yolo_dets:
            if nombre not in trackers:
                trackers[nombre] = TrackerCirculo()
                temporizadores[nombre] = 0.0

        for nombre, tr in list(trackers.items()):
            det, pts = None, None
            if nombre in yolo_dets:
                mejor = max(yolo_dets[nombre], key=lambda d: d[3])
                det = mejor[:3]
                pts = mejor[4]
            result = tr.actualizar(det)
            if result is None:
                temporizadores[nombre] = 0.0
                continue
            hay_objetos = True
            cx, cy, r = result
            if temporizadores[nombre] == 0.0: temporizadores[nombre] = t_act
            color_bgr = get_color_bgr(nombre)
            x_norm = round(cx / fw * 2 - 1, 2)
            y_norm = round(1 - cy / fh * 2, 2)
            z_est  = estimar_z(r, frame.shape)
            if pts is not None:
                overlay = frame.copy()
                cv2.fillPoly(overlay, [pts], color_bgr)
                cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
                cv2.polylines(frame, [pts], True, color_bgr, 2)
            else:
                cv2.circle(frame, (cx, cy), r, color_bgr, 2)
            lbl = f'YOLO: {nombre}'
            lbl_c = f'X:{x_norm:+.1f} Y:{y_norm:+.1f} Z:{z_est}m'
            label_y = max(18, cy - r - 10)
            (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(frame, (cx - r, label_y - th - 4), (cx - r + tw + 4, label_y + 4), (20, 20, 20), -1)
            cv2.putText(frame, lbl, (cx - r + 2, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_bgr, 2)
            cy2 = label_y + th + 6
            (cw, ch), _ = cv2.getTextSize(lbl_c, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
            cv2.rectangle(frame, (cx - r, cy2 - ch - 2), (cx - r + cw + 4, cy2 + 2), (20, 20, 20), -1)
            cv2.putText(frame, lbl_c, (cx - r + 2, cy2), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (210, 210, 210), 1)
            if callback_objeto is not None: callback_objeto(nombre, x_norm, y_norm, z_est, None)
        return frame, hay_objetos

    # Fallback HoughCircles
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (9, 9), 2)
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    raw  = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, dp=HOUGH_DP, minDist=HOUGH_MINDIST, param1=HOUGH_PARAM1, param2=HOUGH_PARAM2, minRadius=HOUGH_MINR, maxRadius=HOUGH_MAXR)
    detecciones_por_color = {}
    if raw is not None:
        for cx, cy, r in np.round(raw[0, :]).astype(int):
            nombre = clasificar_color_circulo(hsv, cx, cy, r)
            if nombre is not None:
                if nombre not in detecciones_por_color or r > detecciones_por_color[nombre][2]:
                    detecciones_por_color[nombre] = (cx, cy, r)
    for nombre in ('Rojo', 'Negro', 'Blanco'):
        if nombre not in trackers:
            trackers[nombre] = TrackerCirculo()
            temporizadores[nombre] = 0.0
    for nombre, tr in trackers.items():
        result = tr.actualizar(detecciones_por_color.get(nombre, None))
        if result is None:
            temporizadores[nombre] = 0.0
            continue
        hay_objetos = True
        cx, cy, r = result
        if temporizadores[nombre] == 0.0: temporizadores[nombre] = t_act
        color_bgr = get_color_bgr(nombre)
        x_norm = round(cx / fw * 2 - 1, 2)
        y_norm = round(1 - cy / fh * 2, 2)
        z_est  = estimar_z(r, frame.shape)
        cv2.circle(frame, (cx, cy), r, color_bgr, 2)
        cv2.circle(frame, (cx, cy), 3, color_bgr, -1)
        if (t_act - temporizadores[nombre]) <= 3.0:
            cv2.putText(frame, 'Calibrando...', (cx - r, max(18, cy - r - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_bgr, 1)
        else:
            lbl = f'Hough: {nombre}'
            lbl_c = f'X:{x_norm:+.1f} Y:{y_norm:+.1f} Z:{z_est}m'
            label_y = max(18, cy - r - 10)
            (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(frame, (cx - r, label_y - th - 4), (cx - r + tw + 4, label_y + 4), (20, 20, 20), -1)
            cv2.putText(frame, lbl, (cx - r + 2, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_bgr, 2)
            cy2 = label_y + th + 6
            (cw, ch), _ = cv2.getTextSize(lbl_c, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
            cv2.rectangle(frame, (cx - r, cy2 - ch - 2), (cx - r + cw + 4, cy2 + 2), (20, 20, 20), -1)
            cv2.putText(frame, lbl_c, (cx - r + 2, cy2), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (210, 210, 210), 1)
            if callback_objeto is not None:
                huella = calcular_huella_hsv(hsv, cx, cy, r)
                callback_objeto(nombre, x_norm, y_norm, z_est, huella)
    return frame, hay_objetos
print('Celda 2 V0.6 lista.')

Celda 2 V0.6 lista.


In [3]:
# Celda 3 - UI y Auto-Optimizacion DEAP  [V0.6]

# ==========================================
# DEAP GA SETUP (Evaluacion offline con buffer)
# ==========================================
if not hasattr(creator, 'FitnessMax'):
    creator.create('FitnessMax', base.Fitness, weights=(1.0,))
if not hasattr(creator, 'Individual'):
    creator.create('Individual', list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
# Genes: [imgsz (múltiplo de 32, de 256 a 640), conf_threshold (0.2 a 0.8), delay_ms (15 a 60)]
toolbox.register('attr_imgsz', random.randrange, 256, 641, 32)
toolbox.register('attr_conf', random.uniform, 0.2, 0.8)
toolbox.register('attr_delay', random.randint, 15, 60)

toolbox.register('individual', tools.initCycle, creator.Individual,
                 (toolbox.attr_imgsz, toolbox.attr_conf, toolbox.attr_delay), n=1)
toolbox.register('population', tools.initRepeat, list, toolbox.individual)

def evaluate_params(individual, buffer_frames, yolo_model):
    if not yolo_model: return (0,)
    imgsz = int(individual[0])
    conf = float(individual[1])
    delay = int(individual[2])
    
    total_time = 0
    detections = 0
    for frame in buffer_frames:
        start = time.time()
        res = yolo_model.predict(frame, imgsz=imgsz, conf=conf, verbose=False)
        total_time += (time.time() - start)
        if res and res[0].boxes is not None and len(res[0].boxes) > 0:
            detections += 1
            
    avg_time_ms = (total_time / len(buffer_frames)) * 1000
    # Castigo si la inferencia tarda más que el delay (bajon de FPS inminente)
    if avg_time_ms > delay:
        penalidad = (avg_time_ms - delay) * 2
    else:
        penalidad = 0
        
    # Queremos ~30 FPS (delay=33)
    fps_score = 100 - abs(delay - 33)
    
    fitness = (detections * 50) + fps_score - penalidad
    return (fitness,)

toolbox.register('mate', tools.cxBlend, alpha=0.5)
toolbox.register('mutate', tools.mutGaussian, mu=0, sigma=10, indpb=0.2)
toolbox.register('select', tools.selTournament, tournsize=3)

def mutacion_personalizada(ind):
    toolbox.mutate(ind)
    # Clamp values
    ind[0] = max(256, min(640, int(ind[0] // 32) * 32))
    ind[1] = max(0.2, min(0.8, ind[1]))
    ind[2] = max(15, min(60, int(ind[2])))
    return ind, 
toolbox.register('mutate', mutacion_personalizada)

# ==========================================
# UI
# ==========================================
class EscanerApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title('Deteccion Profesional  -  V0.6')
        self.geometry('1380x860')
        self.configure(fg_color='#1a1d2e')
        
        # Variables Deteccion
        self.cap = None
        self.escaneando = False
        self.temporizadores = {'Rojo': 0.0, 'Blanco': 0.0, 'Negro': 0.0}
        self.trackers = {'Rojo': TrackerCirculo(), 'Blanco': TrackerCirculo(), 'Negro': TrackerCirculo()}
        self.huellas_memoria = {'Rojo': [], 'Blanco': [], 'Negro': []}
        self.historial_objetos = []
        self.ultimo_intento = {}
        self.COOLDOWN_INTENTO = 1.5
        self.MAX_REGISTROS_POR_COLOR = 2
        
        # Parametros Optimizados (Valores por defecto)
        self.cam_delay_ms = 33
        self.cam_imgsz = 640
        self.cam_conf = 0.45
        
        # Buffer para GP
        self.buffer_frames = collections.deque(maxlen=20)
        self.optimizando = False
        
        self.yolo_model = None
        self._cargar_yolo()
        self._construir_ui()

    def _cargar_yolo(self):
        import os
        from ultralytics import YOLO
        model_path = os.path.join(os.getcwd(), 'models', 'entrenamiento_pelotas', 'weights', 'best.pt')
        if os.path.exists(model_path):
            try:
                self.yolo_model = YOLO(model_path)
            except Exception: pass

    def _construir_ui(self):
        self.lbl_titulo = ctk.CTkLabel(self, text='CONFIGURACION DE CAMARA', font=('Helvetica', 22, 'bold'), text_color='#e8eaf0')
        self.lbl_titulo.pack(pady=(28, 12))
        self.panel_config = ctk.CTkFrame(self, fg_color='transparent')
        self.panel_config.pack(expand=True, fill='both')
        
        self.btn_detectar = ctk.CTkButton(self.panel_config, text='DETECTAR CAMARAS', command=self._accion_buscar)
        self.btn_detectar.pack(pady=10)
        self.frame_lista = ctk.CTkScrollableFrame(self.panel_config, width=700, height=150)
        self.frame_lista.pack(pady=5)
        self.indice_sel = ctk.StringVar(value='-1')
        self.btn_iniciar = ctk.CTkButton(self.panel_config, text='INICIAR DETECCION', state='disabled', command=self._iniciar_camara)
        self.btn_iniciar.pack(pady=40)
        
        self.panel_video = ctk.CTkFrame(self, fg_color='transparent')
        self.lbl_video = ctk.CTkLabel(self.panel_video, text='')
        self.lbl_video.pack(pady=10, padx=10)
        
        self.frame_controles = ctk.CTkFrame(self.panel_video, fg_color='transparent')
        self.frame_controles.pack(pady=8)
        
        self.btn_escaneo = ctk.CTkButton(self.frame_controles, text='Iniciar Escaneo', command=self._toggle_escaneo)
        self.btn_escaneo.grid(row=0, column=0, padx=10)
        
        self.btn_opt_gp = ctk.CTkButton(self.frame_controles, text='Auto-Optimizar (DEAP)', fg_color='#27ae60', hover_color='#2ecc71', command=self._iniciar_optimizacion)
        self.btn_opt_gp.grid(row=0, column=1, padx=10)
        if not self.yolo_model:
            self.btn_opt_gp.configure(state='disabled')
        
        self.btn_limpiar = ctk.CTkButton(self.frame_controles, text='Limpiar Todo', command=self._limpiar_todo)
        self.btn_limpiar.grid(row=0, column=2, padx=10)
        
        self.lbl_gp_status = ctk.CTkLabel(self.panel_video, text='', text_color='#f1c40f', font=('Helvetica', 14, 'bold'))
        self.lbl_gp_status.pack(pady=5)

        self.lista_scroll = ctk.CTkScrollableFrame(self.panel_video, width=300)
        self.lista_scroll.pack(side='right', fill='y', padx=10, pady=10)

    def _accion_buscar(self):
        for w in self.frame_lista.winfo_children(): w.destroy()
        camaras = detectar_camaras_sistema()
        if camaras:
            for idx, etq in camaras:
                ctk.CTkRadioButton(self.frame_lista, text=etq, variable=self.indice_sel, value=str(idx)).pack(anchor='w', pady=5)
            self.indice_sel.set(str(camaras[0][0]))
            self.btn_iniciar.configure(state='normal')

    def _iniciar_camara(self):
        idx = int(self.indice_sel.get())
        if idx == -1: return
        self.panel_config.pack_forget()
        self.panel_video.pack(expand=True, fill='both')
        self.cap = cv2.VideoCapture(idx, cv2.CAP_DSHOW)
        self.escaneando = False
        self._loop_video()

    def _toggle_escaneo(self):
        self.escaneando = not self.escaneando
        if self.escaneando: self.btn_escaneo.configure(text='Pausar Escaneo', fg_color='#d35400')
        else: self.btn_escaneo.configure(text='Reanudar Escaneo', fg_color='#f39c12')

    # -- GP Optimization --
    def _iniciar_optimizacion(self):
        if len(self.buffer_frames) < 10:
            self.lbl_gp_status.configure(text='Error: No hay suficientes frames grabados. Deja el escaneo correr.')
            self.after(3000, lambda: self.lbl_gp_status.configure(text=''))
            return
        self.optimizando = True
        self.escaneando = False
        self.btn_escaneo.configure(text='Reanudar Escaneo', fg_color='#f39c12', state='disabled')
        self.btn_opt_gp.configure(state='disabled')
        self.lbl_gp_status.configure(text='Evolucionando parámetros (DEAP)... Por favor espere.')
        
        # Run DEAP in a thread so UI doesn't completely freeze
        threading.Thread(target=self._run_deap_ga, daemon=True).start()

    def _run_deap_ga(self):
        frames = list(self.buffer_frames)
        toolbox.register('evaluate', evaluate_params, buffer_frames=frames, yolo_model=self.yolo_model)
        pop = toolbox.population(n=10)
        hof = tools.HallOfFame(1)
        try:
            algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=3, halloffame=hof, verbose=False)
            best = hof[0]
            self.cam_imgsz = int(best[0] // 32) * 32
            self.cam_conf = round(float(best[1]), 2)
            self.cam_delay_ms = int(best[2])
            msg = f'¡Optimizacion Exitosa! Resolucion:{self.cam_imgsz}, Conf:{self.cam_conf}, FPS:~{1000//self.cam_delay_ms}'
        except Exception as e:
            msg = f'Error en GP: {e}'
        
        self.after(0, lambda m=msg: self._finalizar_optimizacion(m))

    def _finalizar_optimizacion(self, msg):
        self.optimizando = False
        self.btn_escaneo.configure(state='normal')
        self.btn_opt_gp.configure(state='normal')
        self.lbl_gp_status.configure(text=msg, text_color='#2ecc71')
        self.after(5000, lambda: self.lbl_gp_status.configure(text=''))

    def _loop_video(self):
        if not self.cap or not self.cap.isOpened(): return
        
        if not self.optimizando:
            ret, frame = self.cap.read()
            if ret:
                frame = cv2.flip(frame, 1)
                if self.escaneando:
                    self.buffer_frames.append(frame.copy())
                    frame, _ = procesar_frame_vision(
                        frame, self.temporizadores, self.trackers, 
                        callback_objeto=self.on_objeto_detectado, yolo_model=self.yolo_model,
                        imgsz=self.cam_imgsz, conf=self.cam_conf
                    )
                img = ctk.CTkImage(light_image=Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), size=(920, 630))
                self.lbl_video.configure(image=img)
        
        # Use the GP-evolved delay dynamically!
        self.after(self.cam_delay_ms, self._loop_video)

    def on_objeto_detectado(self, nombre, x, y, z, huella):
        num = sum(1 for o in self.historial_objetos if o['color'] == nombre)
        if num >= self.MAX_REGISTROS_POR_COLOR: return
        self.historial_objetos.append({'color': nombre})
        ctk.CTkLabel(self.lista_scroll, text=f'{nombre} #{num+1}').pack()
        
    def _limpiar_todo(self):
        for w in self.lista_scroll.winfo_children(): w.destroy()
        self.historial_objetos.clear()
        self.temporizadores = {'Rojo': 0.0, 'Blanco': 0.0, 'Negro': 0.0}
        for tr in self.trackers.values(): tr.reiniciar()

print('Celda 3 V0.6: UI lista.')

Celda 3 V0.6: UI lista.


In [4]:
# Celda 4 - Entrenamiento YOLOv8  [V0.6]
from ultralytics import YOLO
from pathlib import Path

def train_model():
    BASE_DIR = Path().absolute()
    DATASET_YAML = BASE_DIR / 'dataset.yaml'
    if not DATASET_YAML.exists():
        print(f'No se encontro: {DATASET_YAML}')
        return
    print('Iniciando entrenamiento ...')
    model = YOLO('yolov8n-seg.pt')
    model.train(task='segment', data=str(DATASET_YAML), epochs=50, imgsz=640, batch=16, project=str(BASE_DIR / 'models'), name='entrenamiento_pelotas', device='cpu')
    print('Modelo guardado.')


In [5]:
# Celda 5 - Punto de entrada  [V0.6]
if __name__ == '__main__':
    app = EscanerApp()
    app.mainloop()